# 🛠️ AI IT Helpdesk Agent — Version 2

**LLM Tool Calling + Agent Loop + Human-in-the-Loop**

### What changed from Version 1?

| Feature | V1 | V2 |
|---------|----|----|  
| Tools | Plain Python functions | `@tool` decorated LangChain tools |
| Routing | Manual `if/else` router | LLM decides which tool to call |
| Flow | Single pass through graph | Agent loop (LLM ↔ Tools) |
| Human-in-the-Loop | ❌ | ✅ Ticket creation requires approval |
| Memory | ❌ | ✅ Checkpointer persists conversation |

### Architecture

```
              USER
               │
               ▼
           ┌───────┐
      ┌───►│ AGENT │
      │    └───┬───┘
      │        │
      │    Needs Tool?
      │    ┌───┴───┐
      │   YES      NO
      │    │        │
      │    ▼        ▼
      │ ┌──────┐   END
      └─┤TOOLS │
        └──────┘
         (with human approval
          for ticket creation)
```

---
## Step 1 — Install Libraries

In [ ]:
%pip install -U langgraph langchain-openai python-dotenv

---
## Step 2 — Imports

Notice the new imports compared to V1:
- `MessagesState` — prebuilt state with message history
- `ToolNode` — automatically executes tool calls
- `tools_condition` — routes based on whether LLM called a tool
- `@tool` decorator — converts functions into LangChain tools
- `MemorySaver` — enables human-in-the-loop by persisting state
- `interrupt` / `Command` — pause and resume for human approval

In [ ]:
import os
import random

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState
)

from langgraph.prebuilt import (
    ToolNode,
    tools_condition
)

from langgraph.checkpoint.memory import MemorySaver

from langgraph.types import (
    interrupt,
    Command
)

from langchain_core.tools import tool

from langchain_core.messages import (
    SystemMessage
)

from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

In [ ]:
load_dotenv()

---
## Step 3 — Create the LLM

In [ ]:
model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=0
)

---
## Step 4 — Define Tools with `@tool`

### V1 vs V2

```
V1: def password_help():        ← plain function, graph manually selects it
V2: @tool def password_help():  ← LangChain tool, LLM decides to call it
```

The `@tool` decorator does three things:
1. Registers the function as a callable tool
2. Uses the **docstring** as the tool description (this is what the LLM reads!)
3. Uses **type hints** to define the input schema

### 4a — Password Tool

In [ ]:
@tool
def password_help() -> str:
    """
    Get troubleshooting instructions
    for password reset issues.

    Use when the user has forgotten their password,
    is locked out, or needs to change their password.
    """

    return """
Password Reset Instructions:

1. Open the company login portal.
2. Click 'Forgot Password'.
3. Verify your registered email.
4. Create a new password.
5. Login again.

If the reset fails, contact IT support.
"""

### 4b — VPN Tool

In [ ]:
@tool
def vpn_help() -> str:
    """
    Get troubleshooting instructions
    for VPN connection problems.

    Use when the user cannot connect to the VPN,
    has a slow VPN, or VPN keeps disconnecting.
    """

    return """
VPN Troubleshooting:

1. Check internet connection.
2. Restart the VPN application.
3. Verify username and password.
4. Disconnect and reconnect.
5. Restart the computer.

If the issue continues,
a support ticket may be required.
"""

### 4c — General IT Tool

In [ ]:
@tool
def general_it_help() -> str:
    """
    Get troubleshooting instructions
    for general IT issues.

    Use for slow computer, software crashes,
    printer problems, network issues, or
    any IT issue not related to VPN or passwords.
    """

    return """
General IT Troubleshooting:

1. Restart the application.
2. Restart the computer.
3. Check network connectivity.
4. Check for software updates.
5. Contact IT support if the issue persists.
"""

### 4d — Create Ticket Tool (with Human-in-the-Loop)

This is the new tool! It uses `interrupt()` to **pause the graph** and ask for human approval before creating a ticket.

```
Agent calls create_ticket
        ↓
  interrupt() pauses graph
        ↓
  Human reviews and approves/rejects
        ↓
  Graph resumes with decision
```

In [ ]:
@tool
def create_ticket(
    issue_summary: str,
    priority: str = "medium"
) -> str:
    """
    Create an IT support ticket for
    unresolved issues.

    Use when the user explicitly asks
    to create a support ticket, or when
    the troubleshooting steps did not
    resolve the issue.

    Args:
        issue_summary: Brief description
            of the issue.
        priority: Ticket priority level.
            Options: low, medium, high.
    """

    # --- Human-in-the-Loop ---
    # This pauses the graph and sends
    # the approval request to the user.

    approval = interrupt(
        {
            "question":
                "Do you approve creating "
                "this support ticket?",

            "summary": issue_summary,
            "priority": priority
        }
    )

    # The graph resumes here after
    # the user responds.

    if approval == "yes":

        ticket_id = f"IT-{random.randint(1000, 9999)}"

        return (
            f"✅ Ticket {ticket_id} created "
            f"successfully!\n"
            f"Summary: {issue_summary}\n"
            f"Priority: {priority}"
        )

    else:

        return (
            "❌ Ticket creation cancelled "
            "by user."
        )

---
## Step 5 — Bind Tools to the LLM

This is the key V2 concept:

```
V1: Graph routes to tools manually
V2: LLM knows about tools and decides which to call
```

`bind_tools()` injects the tool schemas (names, descriptions, parameters) into every LLM call, so the model can output structured tool calls.

In [ ]:
tools = [
    password_help,
    vpn_help,
    general_it_help,
    create_ticket
]

model_with_tools = model.bind_tools(tools)

---
## Step 6 — Understand `MessagesState`

### V1 vs V2 State

```
V1 State (manual):          V2 State (messages):
┌─────────────────┐         ┌─────────────────────┐
│ user_query       │         │ messages             │
│ issue_type       │         │   HumanMessage       │
│ tool_result      │         │   AIMessage          │
│ final_response   │         │   ToolMessage        │
└─────────────────┘         │   AIMessage          │
                            │   ...                │
                            └─────────────────────┘
```

`MessagesState` is a prebuilt LangGraph state with a single key: `messages`.  
Each message is automatically appended (never overwritten), building a conversation history.

In [ ]:
# MessagesState is equivalent to:
#
# class MessagesState(TypedDict):
#     messages: Annotated[list, add_messages]
#
# The add_messages reducer appends new messages
# instead of replacing the list.

print("Using MessagesState from langgraph.graph")

---
## Step 7 — Define the Agent Node

This replaces **both** `classify_issue` and `generate_response` from V1.

The agent node:
1. Prepends a system message (the assistant's persona)
2. Calls the LLM with the full message history
3. The LLM either calls a tool OR responds directly

```
V1: classify_issue → route → tool → generate_response  (4 steps)
V2: agent_node ↔ tools  (loop until done)
```

In [ ]:
SYSTEM_PROMPT = """
You are an IT Helpdesk Assistant.

Your job is to help employees with
IT support requests.

You have access to tools for:
- Password reset issues
- VPN connection problems
- General IT troubleshooting
- Creating support tickets

Always use the appropriate tool first
to get troubleshooting instructions.

Then give a short, clear, professional
answer to the user.

If the issue cannot be resolved with
the troubleshooting steps, offer to
create a support ticket.

Do not invent additional technical details.
"""


def agent_node(state: MessagesState):

    # Prepend system message to give
    # the LLM its persona/instructions

    messages = [
        SystemMessage(content=SYSTEM_PROMPT)
    ] + state["messages"]

    # Call the LLM with tools bound
    response = model_with_tools.invoke(
        messages
    )

    # Return the response — it will be
    # appended to state["messages"]
    return {
        "messages": [response]
    }

---
## Step 8 — Build the Graph

The V2 graph is simpler but more powerful:

```
V1: 5 nodes + conditional routing
V2: 2 nodes + agent loop
```

```
START → agent → tools_condition?
                  ├── has tool calls → tools → agent (loop)
                  └── no tool calls  → END
```

In [ ]:
# Create the graph builder
builder = StateGraph(MessagesState)


# --- Add Nodes ---

builder.add_node(
    "agent",
    agent_node
)

builder.add_node(
    "tools",
    ToolNode(tools)
)


# --- Add Edges ---

# Entry point
builder.add_edge(
    START,
    "agent"
)

# Agent loop:
# If LLM called a tool → go to "tools"
# If LLM responded    → go to END
builder.add_conditional_edges(
    "agent",
    tools_condition
)

# After tools execute, go back to agent
# so LLM can read the result
builder.add_edge(
    "tools",
    "agent"
)

---
## Step 9 — Compile with Memory

`MemorySaver` is required for:
1. **Human-in-the-loop** — persists state while waiting for approval
2. **Conversation memory** — each `thread_id` maintains its own history

```
V1: graph.compile()                    ← no memory
V2: graph.compile(checkpointer=memory) ← with memory
```

In [ ]:
memory = MemorySaver()

helpdesk_graph = builder.compile(
    checkpointer=memory
)

---
## Step 10 — Visualize the Graph

In [ ]:
from IPython.display import (
    Image,
    display
)

display(
    Image(
        helpdesk_graph
        .get_graph()
        .draw_mermaid_png()
    )
)

---
## Step 11 — Helper Function to Run and Print

Since V2 uses message history, we need a helper to run the graph and display results cleanly.

In [ ]:
def run_helpdesk(
    query: str,
    thread_id: str = "default"
):
    """
    Run a query through the helpdesk graph
    and print the results step by step.
    """

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print("=" * 50)
    print(f"User: {query}")
    print("=" * 50)

    # Stream events to see each step
    for step in helpdesk_graph.stream(
        {
            "messages": [
                {"role": "user", "content": query}
            ]
        },
        config
    ):

        for node_name, output in step.items():

            print(f"\n--- Node: [{node_name}] ---")

            for msg in output.get(
                "messages", []
            ):
                msg.pretty_print()

    print("\n" + "=" * 50)

    return config

---
## Step 12 — Test: VPN Issue

The LLM should:
1. Read the query
2. Decide to call `vpn_help` tool
3. Read the tool result
4. Generate a professional response

In [ ]:
run_helpdesk(
    "My VPN is not connecting",
    thread_id="test-vpn"
)

---
## Step 13 — Test: Password Issue

In [ ]:
run_helpdesk(
    "I forgot my password",
    thread_id="test-password"
)

---
## Step 14 — Test: General IT Issue

In [ ]:
run_helpdesk(
    "My laptop is very slow",
    thread_id="test-general"
)

---
## Step 15 — Test: Human-in-the-Loop Ticket Creation

This is the big V2 feature. When the user asks to create a ticket, the graph **pauses** and waits for human approval.

### How it works:

```
1. User asks to create a ticket
2. Agent calls create_ticket tool
3. interrupt() pauses the graph
4. We inspect the paused state
5. We resume with "yes" or "no"
6. Tool completes, agent responds
```

### 15a — Send the ticket request (graph will pause)

In [ ]:
ticket_config = {
    "configurable": {
        "thread_id": "test-ticket"
    }
}

print("Sending ticket request...")
print("(Graph will pause for approval)")
print("=" * 50)

for step in helpdesk_graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":
                    "Create a support ticket. "
                    "My monitor keeps flickering "
                    "and nothing fixes it."
            }
        ]
    },
    ticket_config
):

    for node_name, output in step.items():
        print(f"\n--- Node: [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("\n⏸️  Graph is PAUSED.")
print("Waiting for human approval...")

### 15b — Inspect the paused state

Let's look at what the graph is asking us to approve.

In [ ]:
# Get the current state of the paused thread

state = helpdesk_graph.get_state(
    ticket_config
)

print("Graph status:", state.next)
print()

# Show what the interrupt is asking
for task in state.tasks:
    if task.interrupts:
        for intr in task.interrupts:
            print("Interrupt payload:")
            print(intr.value)

### 15c — Approve the ticket ✅

Resume the graph by sending `Command(resume="yes")`

In [ ]:
print("Approving ticket creation...")
print("=" * 50)

for step in helpdesk_graph.stream(
    Command(resume="yes"),
    ticket_config
):

    for node_name, output in step.items():
        print(f"\n--- Node: [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("\n✅ Ticket flow complete!")

---
## Step 16 — Test: Rejecting a Ticket ❌

Same flow, but we'll say "no" this time.

### 16a — Send the request (graph pauses)

In [ ]:
reject_config = {
    "configurable": {
        "thread_id": "test-reject"
    }
}

print("Sending ticket request...")
print("=" * 50)

for step in helpdesk_graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":
                    "Please create a ticket. "
                    "My email client crashes "
                    "every time I open it."
            }
        ]
    },
    reject_config
):

    for node_name, output in step.items():
        print(f"\n--- Node: [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n⏸️  Graph paused. Waiting for approval...")

### 16b — Reject the ticket

In [ ]:
print("Rejecting ticket creation...")
print("=" * 50)

for step in helpdesk_graph.stream(
    Command(resume="no"),
    reject_config
):

    for node_name, output in step.items():
        print(f"\n--- Node: [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("\n❌ Ticket was rejected.")

---
## Step 17 — Test: Conversation Memory

V2 supports multi-turn conversations within the same thread.  
The agent remembers what was discussed.

In [ ]:
memory_config = {
    "configurable": {
        "thread_id": "test-memory"
    }
}

# Turn 1
print("--- Turn 1 ---")

for step in helpdesk_graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":
                    "My VPN is not connecting."
            }
        ]
    },
    memory_config
):
    for node_name, output in step.items():
        print(f"\n[{node_name}]")
        for msg in output.get("messages", []):
            msg.pretty_print()

In [ ]:
# Turn 2 — follow up on the same thread
print("--- Turn 2 ---")

for step in helpdesk_graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content":
                    "I tried all those steps "
                    "but it still doesn't work. "
                    "Can you create a ticket?"
            }
        ]
    },
    memory_config
):
    for node_name, output in step.items():
        print(f"\n[{node_name}]")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n⏸️  Paused for ticket approval...")

In [ ]:
# Turn 2 continued — approve the ticket
print("--- Approving ---")

for step in helpdesk_graph.stream(
    Command(resume="yes"),
    memory_config
):
    for node_name, output in step.items():
        print(f"\n[{node_name}]")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n✅ Multi-turn conversation complete!")

---
## Step 18 — Inspect Full Message History

Students can see the entire conversation that flowed through the graph.

In [ ]:
# Get final state from the memory thread

final_state = helpdesk_graph.get_state(
    memory_config
)

print("Full message history:")
print("=" * 50)

for i, msg in enumerate(
    final_state.values["messages"]
):
    print(
        f"\n[{i}] {msg.__class__.__name__}"
    )
    print(f"    {msg.content[:100]}..." 
          if len(msg.content) > 100 
          else f"    {msg.content}")

---
## ✅ What We Built — V1 vs V2 Comparison

### Graph Comparison

```
V1 (5 nodes, manual routing):         V2 (2 nodes, agent loop):

START                                 START
  ↓                                     ↓
classify_issue                        agent ◄──┐
  ↓                                     ↓      │
route_issue                          tool call? │
  ┌───┼────┐                          ↓    ↓   │
  ↓   ↓    ↓                         NO   YES  │
PWD  VPN  GEN                         ↓    ↓   │
  └───┼────┘                        END  tools ─┘
      ↓
generate_response
      ↓
     END
```

### Concepts Covered in V2

| Concept | Implementation |
|---------|---------------|
| `@tool` decorator | All 4 tool functions |
| `bind_tools()` | LLM knows about tools |
| `MessagesState` | Message-based state |
| `ToolNode` | Auto-executes tool calls |
| `tools_condition` | Routes agent ↔ tools |
| Agent Loop | `agent → tools → agent → ...` |
| `interrupt()` | Pauses for human approval |
| `Command(resume=)` | Resumes after approval |
| `MemorySaver` | Persists conversation state |
| Multi-turn | Same `thread_id` = same convo |

---

### 🚀 Next Steps (Version 3)

- Add a knowledge base with vector search
- Add escalation logic (auto-escalate after N failed attempts)
- Add sub-graphs for complex multi-step workflows
- Deploy as a web API with LangServe